In [42]:

# Librairies
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors

# intialisation des scaler pour la normalisation des données numériques
scaler = StandardScaler()



# import du df pour le ML version finale a cleaner
df_final = pd.read_csv('C:\\Users\\const\\OneDrive\\Documents\\Formation Data\\Projet 2\\Sprint 4\\Demo_projet2\\data\\df_final.csv')

# import du df pour le ML KNN clean sans keywords
df_ml = pd.read_csv('df_reco_animefiltré.csv')

In [43]:
df_ml.head(5)

,production_countries,id,Action,Animation,Aventure,Comédie,Crime,Documentaire,Drame,Familial,...,Guerre,Histoire,Horreur,Musique,Mystère,Romance,Science-Fiction,Thriller,Téléfilm,Western
0,0,57800,0,1,0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0
1,0,11688,0,1,0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0
2,0,284052,1,0,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,1724,1,0,1,0,0,0,0,0,...,0,0,0,0,0,0,1,0,0,0
4,0,1119878,1,0,0,0,0,0,1,0,...,0,0,0,0,0,0,0,1,0,0


In [51]:
df_ml.columns

Index(['production_countries', 'id', 'Action', 'Animation', 'Aventure',
       'Comédie', 'Crime', 'Documentaire', 'Drame', 'Familial', 'Fantastique',
       'Guerre', 'Histoire', 'Horreur', 'Musique', 'Mystère', 'Romance',
       'Science-Fiction', 'Thriller', 'Téléfilm', 'Western'],
      dtype='object')

In [4]:
df_ml_copy = df_ml.copy()

In [98]:
# on reprend le df de base
df_ml = df_ml_copy

In [ ]:
# selection des colonnes pour le ML

# colonne numérique : 'popularity', 'runtime', 'release_year', 'vote_average', 'vote_count'
cols_num = ['vote_count']

# colonne binaire : toutes les autres colonnes sauf les numériques
cols_binaires = [col for col in df_ml.columns if col not in cols_num]


# Appliquer des poids différents
X_bin = df_ml[cols_binaires].values

X_num = scaler.fit_transform(df_ml[cols_num]) * 0.001
# on rassemble les features et on entrain le mdèle de KNN
# on concatene les deux matrices pour n'en faire qu'une seule

X = np.hstack([X_num, X_bin]) 

In [44]:
df_ml = df_ml.reset_index(drop=True)  # ← ici les index deviennent 0,1,2,3,4...

features = df_ml.drop(columns=['id'])
X = features.values

model = NearestNeighbors(n_neighbors=6, metric='cosine')
model.fit(X)
distances, indices = model.kneighbors(X)

In [53]:
def reco_movie(movie: str):
    # On récupère l'id du film depuis df_final
    id_movie = df_final[df_final['title'] == movie]['id'].values[0]
    
    # On trouve l'indice dans df_ml via l'id
    indice_ml = df_ml[df_ml['id'] == id_movie].index[0]
    
    # On récupère les ids des voisins depuis df_ml
    ids_voisins = df_ml.iloc[indices[indice_ml, 1:]]['id'].values
    
    # On affiche depuis df_final via les ids
    df_reco = df_final[df_final['id'].isin(ids_voisins)]
    return df_reco

In [54]:
reco_movie('Alien, la résurrection')

,backdrop_path,id,title,original_title,overview,popularity,poster_path,release_date,vote_average,vote_count,genres,production_countries,runtime,status,keywords
392,/ySHUoK4utUOiSfCinw81H1TsV0E.jpg,1241470,Osiris,Osiris,Une équipe de commandos des forces spéciales e...,15.4577,/xIwzeCCShj5N8dSHPDMkKvzi1Al.jpg,2025-07-25,5.991,229,"['Science-Fiction', 'Action', 'Horreur']",US,127,Released,"['spacecraft', 'special forces', 'absurd']"
964,/63eAFgJtN6ZTyFFzlAOoZLhlYqc.jpg,36648,Blade : Trinity,Blade: Trinity,À l'aide d'une manipulation d'image aussi géni...,6.6123,/bA9HTKZljF80Q0tmtYcjbQ0viwg.jpg,2004-12-08,5.900,4282,"['Action', 'Horreur', 'Science-Fiction']",US,106,Released,"['martial arts', 'loss of loved one', 'vampire..."
3418,/hIHtyIYgBqHybOgUdoAmveipuiO.jpg,703745,Peur Bleue 3,Deep Blue Sea 3,"Emma Collins, une éminente biologiste marine, ...",2.6811,/84MBYqjgfY1CBpxAulF9MdIe6E4.jpg,2020-08-25,5.556,349,"['Action', 'Horreur', 'Science-Fiction']",US,99,Released,"['ocean', 'boat', 'shark attack', 'animal atta..."
3431,/fA12BDNnyTATuBw4xSKDcqDdi9p.jpg,399894,Daylight's End,Daylight's End,"Dans le futur, la Terre a été dévastée par une...",2.6739,/sNDodM6si97uhiPhf5dTM6n6aDj.jpg,2016-04-16,5.781,194,"['Action', 'Horreur', 'Science-Fiction']",US,95,Released,['plague']
6502,/uLZ3NgKf2qLjSPyqNIEYYMABboB.jpg,16139,Phantasm III - Le seigneur de la mort,Phantasm III: Lord of the Dead,"Apres quatorze ans de lutte sans merci, Mike e...",1.3420,/mOHDbHVjQyT8a4OhvMKt8R6fD8j.jpg,1994-05-06,5.808,281,"['Horreur', 'Science-Fiction', 'Action']",US,91,Released,"['zombie', 'mausoleum', 'tall man', 'mortuary']"


In [55]:
reco_movie('Doctor Strange')

,backdrop_path,id,title,original_title,overview,popularity,poster_path,release_date,vote_average,vote_count,genres,production_countries,runtime,status,keywords
2,/3zvZ699gMW2RhWc0GisIukzq0Ls.jpg,284052,Doctor Strange,Doctor Strange,"Le neurochirurgien Stephen Strange, en quête d...",12.0140,/7wZ7mx7tY5SgflQKuJmQvwu3wGm.jpg,2016-10-25,7.419,23352,"['Fantastique', 'Aventure', 'Action']",US,115,Released,"['magic', 'superhero', 'training', 'based on c..."
575,/zMrk2G3XsnfYKiIp1NEfdtvDyBH.jpg,337401,Mulan,Mulan,Lorsque l’Empereur de Chine publie un décret s...,8.0886,/zJydDjqv56pRWiQhVFpzlsQdzjR.jpg,2020-09-04,6.800,7021,"['Aventure', 'Fantastique', 'Action']",US,115,Released,"['martial arts', 'hero', 'china', 'fake identi..."
860,/4rICId8WQBhOIzl7iZCVvc0QoJW.jpg,9387,Conan le barbare,Conan the Barbarian,"Encore enfant, Conan assiste impuissant au mas...",6.5589,/qwnXGJCAMYrJ4KZo5d5zgCeMEjc.jpg,1982-03-16,6.803,2771,"['Aventure', 'Fantastique', 'Action']",US,129,Released,"['based on novel or book', 'gladiator', 'fight..."
2173,/9Lm94WVtV4HPxr7NWNlwkhLKSeH.jpg,23047,Le dernier des templiers,Season of the Witch,"Après des années de croisade en Terre sainte, ...",3.8327,/A0y0wscUGKoPOwMQZjfjCQdrBye.jpg,2011-01-07,5.525,2631,"['Aventure', 'Fantastique', 'Action']",US,95,Released,"['witch', 'hero', 'ambush', 'fight', 'ritual',..."
3080,/syGl3udgPqjCYrs6pzbcV0Tb2jk.jpg,522417,Le Roi Scorpion : Le Livre des âmes,Scorpion King: Book of Souls,"Dans l’Égypte ancienne, le sanguinaire seigneu...",2.8655,/eoJDNOT0xw3rkYQxzJwntxNa6Tq.jpg,2018-10-23,5.900,316,"['Action', 'Aventure', 'Fantastique']",US,102,Released,"['warlord', 'sister', 'prequel', 'relic', 'fem..."


In [49]:
reco_movie('La Reine des neiges II')

501                       Le Roi lion 3 : Hakuna matata
532                                 Le Monstre des mers
5246    La Cour de récré - Rentrée en classe supérieure
5275            Scooby-Doo !  Aventures en Transylvanie
5458                        Les aventures de Mark Twain
Name: title, dtype: object